
# Mass completeness limit in SDSS-like photometric surveys

Measures the 95% stellar mass completeness threshold for SDSS-like photometry.
Mocks a population of 150 star-forming and passive galaxies spanning
log M* [7-12] at z=0.1, injects realistic photometric noise, and measures
below which stellar mass more than 5% of sources drop below detection limit.
Critical for constructing mass-limited galaxy samples and understanding
survey selection effects.

Reference: Blanton et al. 2003, ApJ, 594, 186 (SDSS photometry);
Bell et al. 2003, ApJS, 149, 289 (stellar mass completeness).


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()
bands = ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"]
obs = tengri.Observation(photometry=tengri.Photometry.from_names(bands))

# Build model
model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "tsnorm",
        "log_peak_sfr": tengri.Uniform(-1.0, 2.5),
        "peak_lbt_gyr": tengri.Uniform(0.5, 12.0),
        "width_gyr": tengri.Uniform(0.3, 5.0),
        "skew": tengri.Uniform(-3.0, 3.0),
        "trunc": tengri.Uniform(1.0, 10.0),
        "logzsol": tengri.Uniform(-2.0, 0.2),
    },
    dust={
        "type": "two_component",
        "tau_bc": tengri.Uniform(0.0, 2.0),
        "tau_diff": tengri.Uniform(0.0, 1.5),
        "slope": tengri.Fixed(-0.7),
    },
    redshift=tengri.Fixed(0.1),
)

# Generate population
key = jax.random.PRNGKey(123)
log_mass_array = []
detected_array = []

n_gal = 150
mass_bins = np.linspace(7, 12, 11)

for i in range(n_gal):
    key, subkey = jax.random.split(key)
    params = model.spec.sample(subkey)
    # Assign log-mass uniformly
    log_mass = np.random.uniform(7, 12)
    # Heuristic: older/more massive galaxies have higher SFR peak
    params["sfh_tsnorm_log_peak_sfr"] = 0.5 + (log_mass - 9.0) * 0.2
    params["sfh_tsnorm_peak_lbt_gyr"] = 3.0 + (log_mass - 9.0) * 1.0

    # Predict photometry and inject noise
    fluxes = np.asarray(model.predict_photometry(params))
    snr = 20.0  # Nominal SNR
    noise = fluxes / snr
    fluxes_obs = fluxes + np.random.normal(0, 1, size=len(fluxes)) * noise
    # Detection: require SNR > 3 in at least 3 bands
    detected = np.sum(fluxes_obs / noise > 3) >= 3
    log_mass_array.append(log_mass)
    detected_array.append(detected)

log_mass_array = np.array(log_mass_array)
detected_array = np.array(detected_array)

# Compute completeness
completeness = []
mass_centers = (mass_bins[:-1] + mass_bins[1:]) / 2
for j in range(len(mass_bins) - 1):
    mask = (log_mass_array >= mass_bins[j]) & (log_mass_array < mass_bins[j + 1])
    if np.sum(mask) > 0:
        comp = np.sum(detected_array[mask]) / np.sum(mask)
        completeness.append(comp)
    else:
        completeness.append(0.0)

completeness = np.array(completeness)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(mass_centers, completeness, "o-", lw=2.0, ms=8, color="C0", label="SDSS ugriz")
ax.axhline(0.95, color="red", ls="--", lw=1.5, label="95% completeness")
ax.axhline(0.90, color="orange", ls=":", lw=1.5, label="90% completeness")

# Find 95% completeness mass
idx_95 = np.argmin(np.abs(completeness - 0.95))
m_complete_95 = mass_centers[idx_95]
ax.axvline(m_complete_95, color="red", ls="--", alpha=0.3, lw=1.0)

ax.set_xlabel(r"Stellar Mass log$_{10}$(M$_\odot$)")
ax.set_ylabel("Completeness")
ax.set_ylim([0.75, 1.05])
ax.set_xlim([7, 12])
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig("plot_usecase_mass_completeness.png", dpi=150, bbox_inches="tight")